#Building a function to build complexes


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import ipywidgets as widgets
from IPython.display import display
from scipy.spatial import cKDTree # Optimized import

# --- 1. Define Torus Parameters and Generate Points ---
R = 5  # Major radius
r = 2  # Minor radius
n_points = 150  # Number of random points (increased for performance demo)

u = 2 * np.pi * np.random.rand(n_points)
v = 2 * np.pi * np.random.rand(n_points)

x = (R + r * np.cos(v)) * np.cos(u)
y = (R + r * np.cos(v)) * np.sin(u)
z = r * np.sin(v)

# --- 2. Add Noise to the Data ---
noise_scale = 0.5  # Adjust this value to control the amount of noise
x_noisy = x + np.random.normal(0, noise_scale, n_points)
y_noisy = y + np.random.normal(0, noise_scale, n_points)
z_noisy = z + np.random.normal(0, noise_scale, n_points)

# --- 3. Optimized Connection Function (using cKDTree) ---
def connect_points_by_radius(x, y, z, connection_radius):
    points = np.column_stack((x, y, z))
    tree = cKDTree(points)
    pairs = tree.query_pairs(connection_radius)
    connections = list(pairs)
    return connections

# --- 3.1 Optimized Triangle Finding Function ---
def find_triangles_from_connections(connections, n_points):
    adjacency_list = {i: set() for i in range(n_points)}
    for p1, p2 in connections:
        adjacency_list[p1].add(p2)
        adjacency_list[p2].add(p1)

    unique_triangles = set()
    for u, v in connections:
        common_neighbors = adjacency_list[u].intersection(adjacency_list[v])
        for w in common_neighbors:
            triangle = tuple(sorted((u, v, w)))
            unique_triangles.add(triangle)
    return sorted(list(unique_triangles))

# --- Pre-generate a base sphere mesh for plotting (low resolution for performance) ---
phi, theta = np.mgrid[0:np.pi:10j, 0:2*np.pi:10j] # Reduced resolution
x_base_sphere = np.sin(phi) * np.cos(theta)
y_base_sphere = np.sin(phi) * np.sin(theta)
z_base_sphere = np.cos(phi)

# --- 4. Define Interactive Visualization Function ---
def update_plot_with_radius(connection_radius_val, elev, azim):
    plt.close('all')

    connections = connect_points_by_radius(x_noisy, y_noisy, z_noisy, connection_radius_val)
    triangles = find_triangles_from_connections(connections, n_points)

    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(111, projection='3d')

    # Set viewpoint
    ax.view_init(elev=elev, azim=azim)

    # Plot spheres at noisy points, scaled by connection_radius_val
    if n_points > 0: # Ensure there are points to plot
        # Create a dummy legend entry for spheres
        ax.plot_surface(x_base_sphere * connection_radius_val + x_noisy[0],
                        y_base_sphere * connection_radius_val + y_noisy[0],
                        z_base_sphere * connection_radius_val + z_noisy[0],
                        color='blue', alpha=0.01, linewidth=0, label='Noisy Points (Spheres)')
        for idx in range(n_points):
            # Scale and translate the base sphere mesh for each point
            sphere_x = x_base_sphere * connection_radius_val + x_noisy[idx]
            sphere_y = y_base_sphere * connection_radius_val + y_noisy[idx]
            sphere_z = z_base_sphere * connection_radius_val + z_noisy[idx]
            ax.plot_surface(sphere_x, sphere_y, sphere_z, color='blue', alpha=0.01, linewidth=0)

    # Add a scatter plot for individual markers on top of the spheres
    ax.scatter(x_noisy, y_noisy, z_noisy, c='darkblue', s=20, alpha=0.8, label='Noisy Points (Markers)')

    # Plot connections
    if connections:
        ax.plot([], [], [], color='red', linewidth=1.5, alpha=0.5, label='Connections')
        for i, j in connections:
            x_pair = [x_noisy[i], x_noisy[j]]
            y_pair = [y_noisy[i], y_noisy[j]]
            z_pair = [z_noisy[i], z_noisy[j]]
            ax.plot(x_pair, y_pair, z_pair, color='red', linewidth=1.5, alpha=0.5)

    # Plot triangles as 3D surfaces
    if triangles:
        ax.plot_trisurf(x_noisy, y_noisy, z_noisy, triangles=np.array(triangles), color='cyan', alpha=0.8, label='Triangles')
    else:
        ax.plot([], [], [], color='cyan', alpha=0.4, label='Triangles') # Dummy for legend

    ax.set_xlabel('X Noisy')
    ax.set_ylabel('Y Noisy')
    ax.set_zlabel('Z Noisy')
    ax.set_title(f'Noisy Torus Points with Dynamic Connections & Triangles (Radius: {connection_radius_val:.2f})')

    max_range_noisy = np.array([
        x_noisy.max() - x_noisy.min(),
        y_noisy.max() - y_noisy.min(),
        z_noisy.max() - z_noisy.min()
    ]).max() / 2.0
    mid_x_noisy = (x_noisy.max() + x_noisy.min()) * 0.5
    mid_y_noisy = (y_noisy.max() + y_noisy.min()) * 0.5
    mid_z_noisy = (z_noisy.max() + z_noisy.min()) * 0.5

    ax.set_xlim(mid_x_noisy - max_range_noisy, mid_x_noisy + max_range_noisy)
    ax.set_ylim(mid_y_noisy - max_range_noisy, mid_y_noisy + max_range_noisy)
    ax.set_zlim(mid_z_noisy - max_range_noisy, mid_z_noisy + max_range_noisy)

    ax.legend()
    plt.show()

# --- 5. Create and Display Interactive Slider ---
radius_slider = widgets.FloatSlider(
    value=1.5,
    min=0.1,
    max=5.0,
    step=0.1,
    description='Connection Radius:',
    continuous_update=True,
    orientation='horizontal',
    readout=True,
    readout_format='.1f',
)

elev_slider = widgets.IntSlider(
    value=30,
    min=-90,
    max=90,
    step=5,
    description='Elevation:',
    continuous_update=True,
    orientation='horizontal',
    readout=True,
)

azim_slider = widgets.IntSlider(
    value=-60,
    min=-180,
    max=180,
    step=5,
    description='Azimuth:',
    continuous_update=True,
    orientation='horizontal',
    readout=True,
)

# Link the sliders to the update function and display the output widget
output_widget = widgets.interactive_output(update_plot_with_radius, {
    'connection_radius_val': radius_slider,
    'elev': elev_slider,
    'azim': azim_slider
})
display(widgets.VBox([radius_slider, elev_slider, azim_slider]), output_widget)

print("All optimized functionalities are now combined into a single interactive cell.")

#Simplicial Complex Visualize


In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import gudhi as gd
# from google.colab import drive
# drive.mount('/content/drive')

# 1. Load coordinates from CSV
df = pd.read_csv("/workspaces/Cellular-TDA/data/Old Data/centroids Epha2a hom_table.csv")
points = df[['Center of the object_0', 'Center of the object_1', 'Center of the object_2']].values


# 2. Generate Simplicial Complex (Alpha Complex)
# This creates simplices based on a distance threshold (alpha)
alpha_complex = gd.AlphaComplex(points=points)
simplex_tree = alpha_complex.create_simplex_tree()

# Extract triangles (2-simplices) for visualization
# We filter by a small alpha value to keep only "close" connections
alpha_value = 1500 # Increased alpha_value to capture more triangles

# Get all 2-simplices and their filtration values
all_2_simplices = [s for s in simplex_tree.get_skeleton(2) if len(s[0]) == 3]

# Determine the maximum alpha value for existing triangles to guide user
max_triangle_alpha = 0
if all_2_simplices:
    max_triangle_alpha = max(s[1] for s in all_2_simplices)
    print(f"Maximum alpha value for any triangle in the complex: {max_triangle_alpha:.2f}")

# Filter triangles based on alpha_value
filtered_triangles_list = [s[0] for s in all_2_simplices if s[1] <= alpha_value]

if filtered_triangles_list:
    triangles = np.array(filtered_triangles_list, dtype=int)
else:
    # If no triangles are found, create an empty array with the correct shape (0, 3)
    # This prevents the IndexError when trying to access triangles[:, 0]
    triangles = np.empty((0, 3), dtype=int)
    print(f"No triangles found for alpha_value <= {alpha_value}. Consider increasing alpha_value.")

# 3. Visualize with Plotly Mesh3d
fig = go.Figure(data=[
    go.Mesh3d(
        x=points[:, 0],
        y=points[:, 1],
        z=points[:, 2],
        i=triangles[:, 0], # Index of first vertex of each triangle
        j=triangles[:, 1], # Index of second vertex
        k=triangles[:, 2], # Index of third vertex
        opacity=0.5,
        color='cyan'
    )
])

fig.update_layout(title="3D Simplicial Complex from CSV")
fig.show()

In [ ]:
# 1. Load coordinates from CSV
df = pd.read_csv("/workspaces/Cellular-TDA/data/Old Data/centroids control_table.csv")
points = df[['Center of the object_0', 'Center of the object_1', 'Center of the object_2']].values


# 2. Generate Simplicial Complex (Alpha Complex)
# This creates simplices based on a distance threshold (alpha)
alpha_complex = gd.AlphaComplex(points=points)
simplex_tree = alpha_complex.create_simplex_tree()

# Extract triangles (2-simplices) for visualization
# We filter by a small alpha value to keep only "close" connections
alpha_value = 1000 # Increased alpha_value to capture more triangles

# Get all 2-simplices and their filtration values
all_2_simplices = [s for s in simplex_tree.get_skeleton(2) if len(s[0]) == 3]

# Determine the maximum alpha value for existing triangles to guide user
max_triangle_alpha = 0
if all_2_simplices:
    max_triangle_alpha = max(s[1] for s in all_2_simplices)
    print(f"Maximum alpha value for any triangle in the complex: {max_triangle_alpha:.2f}")

# Filter triangles based on alpha_value
filtered_triangles_list = [s[0] for s in all_2_simplices if s[1] <= alpha_value]

if filtered_triangles_list:
    triangles = np.array(filtered_triangles_list, dtype=int)
else:
    # If no triangles are found, create an empty array with the correct shape (0, 3)
    # This prevents the IndexError when trying to access triangles[:, 0]
    triangles = np.empty((0, 3), dtype=int)
    print(f"No triangles found for alpha_value <= {alpha_value}. Consider increasing alpha_value.")

# 3. Visualize with Plotly Mesh3d
fig = go.Figure(data=[
    go.Mesh3d(
        x=points[:, 0],
        y=points[:, 1],
        z=points[:, 2],
        i=triangles[:, 0], # Index of first vertex of each triangle
        j=triangles[:, 1], # Index of second vertex
        k=triangles[:, 2], # Index of third vertex
        opacity=0.5,
        color='cyan'
    )
])

fig.update_layout(title="3D Simplicial Complex from CSV")
fig.show()

In [ ]:
import pandas as pd
import numpy as np
import gudhi as gd
import plotly.graph_objects as go
from ipywidgets import interact, FloatSlider

# 1. Load coordinates
df = pd.read_csv("/workspaces/Cellular-TDA/data/Old Data/centroids control_table.csv")
points = df[['Center of the object_0', 'Center of the object_1', 'Center of the object_2']].values

# 2. Pre-compute Simplicial Complex (Alpha Complex)
# We do this outside the function for better performance
alpha_complex = gd.AlphaComplex(points=points, )
simplex_tree = alpha_complex.create_simplex_tree()

# Pre-extract all 2-simplices and their filtration values (alpha^2)
all_2_simplices = [(s[0], s[1]) for s in simplex_tree.get_skeleton(2) if len(s[0]) == 3]

# Define the update function for the slider
def update_plot(alpha_value):
    # Filter triangles based on alpha_value
    filtered_triangles = [s[0] for s in all_2_simplices if s[1] <= alpha_value]

    if filtered_triangles:
        triangles = np.array(filtered_triangles, dtype=int)
    else:
        triangles = np.empty((0, 3), dtype=int)

    # 3. Visualize with Plotly
    fig = go.Figure(data=[
        go.Mesh3d(
            x=points[:, 0], y=points[:, 1], z=points[:, 2],
            i=triangles[:, 0] if len(triangles) > 0 else [],
            j=triangles[:, 1] if len(triangles) > 0 else [],
            k=triangles[:, 2] if len(triangles) > 0 else [],
            opacity=0.5,
            color='cyan'
        ),
        # Add scatter points so the nodes remain visible even without triangles
        go.Scatter3d(
            x=points[:, 0], y=points[:, 1], z=points[:, 2],
            mode='markers',
            marker=dict(size=2, color='black', opacity=0.8)
        )
    ])

    fig.update_layout(
        title=f"3D Simplicial Complex (Alpha: {alpha_value})",
        scene=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Z'),
        margin=dict(l=0, r=0, b=0, t=40), width= 800, height=600
    )
    fig.show()

# Determine a reasonable range for the slider
max_alpha = max([s[1] for s in all_2_simplices]) if all_2_simplices else 1000
print(max_alpha)
# Create the interactive slider
interact(update_plot, alpha_value=FloatSlider(
    value=0,
    min=0,
    max=16227,
    step=500,
    description='Alpha Value:'
));
